<a href="https://colab.research.google.com/github/HarrisSmyrillis/Transport-Hackathon-Team-Innovators/blob/main/Beyond_The_Prompt_Workshop_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Beyond the Prompt workshop

In this workshop, we are going to build and play a simple game called "20 Questions." This game uses Python and the Google ADK framework.

**How the game works**-

You will play the "20 Questions" game with an AI. You think of a common noun (a word). The AI will ask up to 20 yes/no questions. You will answer "yes" or "no" and the AI will try and guess your word.

## Part 1: Install Required Packages


The Google Agent Development Kit (ADK) is the framework that helps us build structured AI-powered applications. To get started, we need to install the necessary Python packages

In [ ]:
# Install Google ADK
!pip install -q google-adk

The Google ADK relies on Python's built-in `asyncio` library for handling asynchronous processing (tasks that don't need to wait for each other to finish).


`asyncio` was first introduced in Python 3.4. You don't need to be an expert in asyncio for this workshop. But if you want to find out more, here is a good [conceptual overview of asyncio](https://docs.python.org/3/howto/a-conceptual-overview-of-asyncio.html#a-conceptual-overview-of-asyncio), which you can read after the workshop.

In [ ]:
import asyncio

# Enabling nested asyncio - This is a hack for Colab only
# (Because Colab already runs in an asyncio event loop)
!pip install nest_asyncio

import nest_asyncio
nest_asyncio.apply()

In [ ]:
# Import some common utility classes
import os
import json

# Import a couple of utility functions for Colab

from IPython.display import Markdown, display

We will import more dependencies as we go.

## Part 2: Supply Gemini API Key for ADK


We need your Gemini API Key for the ADK to connect to the model.

Get the API Key from Google AI Studio:

1. Go to: `https://aistudio.google.com/`

2. After logging in, select **"Get API key"** (at the bottom left corner).

3. Then, on the new page, select **"Create API key"** (at the top right corner).

4. Choose a Google Cloud project or let AI Studio create one for you.


Then, copy and paste the key as a Colab secret:

1. Copy the key from AI Studio.
2. Click on the **"key" icon (🔑)** in the left-hand sidebar of Colab (this opens the Secrets panel).
3. Add a new secret and name it **`GEMINI_API_KEY`**.
4. Paste the key's value in and ensure access is enabled for this notebook.

Run the following code to read the secret and set the environment variables, so this Colab notebook can use your Gemini API key.

In [ ]:
# Using userdata is a safe way to handle keys in Colab
from google.colab import userdata

# Get the value of GOOGLE_API_KEY from Colab userdata
# And set the environment variable GOOGLE_API_KEY
os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')

# Configure ADK to use Gemini API directly (not Vertex AI)
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

GEMINI_MODEL= "gemini-2.5-flash"

## Part 3: Simple Agent




In this step, we are going to create a simple Agent to check the user's input and make sure it's a valid word for our game.

**What is a valid word?**

For this "20 Questions" game, a valid word should be: a **common noun** (like the name of an object, an animal, or a concept), and **not too obscure** or specialized.

Checking these criteria with traditional rule-based logic is difficult Therefore, we will an agent to perform this smart validation for us.

### Task 3.1: Define the validation agent


Example code: https://gist.github.com/yoyu777/6271a099e10683dd783bae8e5a7b81c5

In [ ]:
# ADK uses pydantic for json schema validation
# Import a couple of utility classes here
from pydantic import BaseModel, Field

# Import the LlmAgent class from ADK
from google.adk.agents import LlmAgent

# Specify the output schema
# Valid output should have two fields:
# `is_valid`, which is a boolean type and indicates if the input is a valid object for the game.
# `reason`, which is a string that explains why the AI thinks the input is valid or not.
class ValidationOutput(BaseModel):
    is_valid: [...complete the code here…]
    reason: [...complete the code here…]

# Define the validation agent here
validation_agent = [...complete the code here…]

### Task 3.2: Create the validation session

A user "session" is like a conversation with an agent. Each session has its own context and state.

Example code: https://gist.github.com/yoyu777/f4a7227d16abe1b7a9e8cd9275c7f1a0

In [ ]:
# We will use the InMemorySessionService for this session
# It means the session data only lives in the computer's memory
# while the program is running.
from google.adk.sessions import InMemorySessionService

session_service = InMemorySessionService()

app_name = "ValidationAgent"
user_id="test_user"

async def create_validation_session():
  # Create a session
  # `create_session` is an asynchronous function,
  # therefore, must be awaited
  validation_session = [...complete the code here…]
  return validation_session

# Await the creation of the session properly
# Since this is top-level code, we use asyncio.run to execute the async function
validation_session = [...complete the code here…]

display("Validation session initialised")

### Task 3.3: Create the validation agent runner

Example code: https://gist.github.com/yoyu777/f6b0986787b77d960a227df8f634143f

In [ ]:
from google.adk.runners import Runner

validation_agent_runner = [...complete the code here…]

### Task 3.4: Create the validation UI in Colab

For this task, simply **run this block of code**. You are **not expected to edit** anything in this block, as our focus in this workshop is on ADK-related content.

**`ipywidgets`** is a library specifically used for creating user interfaces (UI) within Colab notebooks. If you are interested, you can read more about it in your own time: https://pypi.org/project/ipywidgets/

In [ ]:
import ipywidgets as widgets

# Function to create all the UI widgets we need
def create_validation_ui():
  # text to prompt the user to think of a word
  prompt_label = widgets.Label("Think of a word:")

  # text input field
  text_input = widgets.Text(
      value='',
      placeholder='Enter your word here',
      description='Your Word:',
      disabled=False
  )

  # confirm button
  confirm_button = widgets.Button(
      description='Confirm',
      disabled=False,
      button_style='',
      tooltip='Click to confirm your word',
      icon='check'
  )

  return prompt_label, text_input, confirm_button

prompt_label, text_input, confirm_button=create_validation_ui()

# Arrange the widgets in a vertical box
ui = widgets.VBox([prompt_label, text_input, confirm_button])

# Display the UI
display(ui)

You'll notice that when we hit the **"Confirm"** button, it doesn't do anything at the moment. This is because the button is not bound to any code yet.

In this next step, **we are going to create the callback function** for the button click.

**What is a Callback?**

* A callback function is the specific block of code that runs when the button is clicked.
* We'll define this function to handle the word entered by the user and send it to our validation Agent.

### Task 3.5: Call the agent upon user input

We need to pass a function to the button widget. The goal is that when the user types a word and presses the **"Confirm"** button, this function will execute and **call our Agent** to perform the validation.

Example code: https://gist.github.com/yoyu777/51c3b5594f5beba5246eb559911cda42

In [ ]:
# This is a utility module for ADK data structures
from google.genai import types

# This function needs to be **asynchronous** (`async def`),
# as we need to **await** the responses to be produced.

async def validate_input(user_input: str):

  try:

    print(f'validating input: {user_input}')

    # `run_async` will return an asynchronous Generator,
    # which may produce various responses (stream)
    # Therefore, we must use an `async for ... in ...` loop
    # to correctly process the streamed output.

   [...complete the code here…]

  except Exception as e:
      display(f"Error during validation: {e}")
      return None

Next, we will link everything together to complete the validation agent.

In [ ]:
# Create a new session
validation_session = asyncio.run(create_validation_session())

# Re-create the UI widgets
prompt_label, text_input, confirm_button=create_validation_ui()

# Callback function when the button is pressed
# It needs to be a non-asychonorous function, and accepts one argument (the button object itself)
# Eventhough the argument is not needed in this case.
def on_confirm_button_clicked(b):
    try:
      # Schedule the async function to run and get its result
      validation_result = asyncio.run(validate_input(text_input.value))
      if validation_result:
        display(Markdown(f'**{validation_result.get("reason")}**')) # Display the parsed Pydantic object
    except Exception as e:
      display(f"Error during button click: {e}")

# Pass the function to
confirm_button.on_click(on_confirm_button_clicked)

# Arrange the widgets in a vertical box
ui = widgets.VBox([prompt_label, text_input, confirm_button])

# Display the UI
display(ui)

NameError: name 'create_validation_session' is not defined

## Part 4: Use of the multi-agent pattern

In this part, **we are going to define two specialized Agents** to play the "20 Questions" game.

**Agent 1: The Guesser**

This Agent is specialized in **making guesses**. At the start of each round, it will try to make a guess and **assess if it is confident enough**.

**Agent 2: The Questioner**

The Questioner Agent's job is to **formulate a "yes/no" question** to ask the player, helping gather the necessary information.


The results produced by both agent will be stored in session context with specified **`output_key`**.

### Task 4.1: Define the Guessing Agent

In [ ]:
class GuessOutput(BaseModel):
    guess: str = Field(..., description="The final guess for what the user is thinking of")
    confidence: int = Field(..., description="Confidence level (1-10) in this guess")
    reasoning: str = Field(..., description="Explanation of why this is the best guess. Summarise in less than 20 words.")

guessing_agent = LlmAgent(
    name="guessing_agent",
    model=GEMINI_MODEL,
    instruction="You are an expert at making educated guesses in 20 Questions game",
    description="""You analyze all the information gathered from previous questions and answers
    to make the best possible guess about what the user is thinking of.
    Consider:
    - All yes/no answers received so far
    - Common objects that fit the criteria
    - Probability and likelihood of different possibilities
    - The specificity needed based on question count
    Make confident, well-reasoned guesses when you have enough information.
    """,
    output_schema=GuessOutput,
    output_key="guess_output"
)



### Task 4.2: Define the Asking Agent

In [ ]:
class QuestionOutput(BaseModel):
    question: str = Field(..., description="A strategic yes/no question to ask")
    reasoning: str = Field(..., description="Explanation of why you ask this question. Summarise in less than 20 words.")

asking_agent = LlmAgent(
    name="asking_agent",
    model=GEMINI_MODEL,
    instruction="You are an expert at asking strategic yes/no questions in 20 Questions game",
    description="""You specialize in asking the most effective yes/no questions to narrow down possibilities.
    Your goal is to eliminate as many possibilities as possible with each question.
    Consider categories like:
    - Living vs non-living
    - Size (bigger/smaller than a breadbox)
    - Location (indoor/outdoor, natural/man-made)
    - Function or use
    - Physical properties (color, texture, material)
    Ask questions that divide the remaining possibilities roughly in half.
    Avoid overly specific questions early in the game.
    """,
    output_schema=QuestionOutput,
    output_key="question_output"
)

## Part 5: Custom Routing Agent

We will create a custom routing agent by extending the `BaseAgent` class.

This root agent will act as our **game's manager**. Its job is to:

1.  First, call the Guessing Agent.
2.  Based on the **confidence score** returned by the Guesser, it will make a decision:
    * **If confident:** Decide to make a guess.
    * **If not confident:** Decide to continue the game by calling the Questioner Agent to formulate a new "yes/no" question.

In [ ]:
from google.adk.agents import BaseAgent
from google.adk.events import Event
from google.adk.agents.invocation_context import InvocationContext
from typing import AsyncGenerator

# To create the custom agent, we will extend the `BaseAgent` class
class RootAgent(BaseAgent):
    guessing_agent: LlmAgent
    asking_agent: LlmAgent

    # class constructor
    def __init__(self, name: str, guessing_agent: LlmAgent, asking_agent: LlmAgent):

        # calling the parent constructor
        super().__init__(
            name=name,
            guessing_agent=guessing_agent,
            asking_agent=asking_agent
        )

    # utility function
    def create_text_response_event(self, response: str, invocation_id: str) -> Event:
        event = Event(
            content=types.Content(
                role=self.name,
                parts=[types.Part(text=response)]
            ),
            author=self.name,
            invocation_id=invocation_id
        )
        return event

    # overriding run async implementation (of the BaseAgent class)
    async def _run_async_impl(self, ctx: InvocationContext) -> AsyncGenerator[Event, None]:
        invocation_id = ctx.invocation_id
        print(f"{self.name} started running")

        async for event in self.guessing_agent.run_async(ctx):
            yield event

        guess_output = ctx.session.state.get("guess_output", None)
        confidence = guess_output.get("confidence") if guess_output else None

        if guess_output is None or confidence is None:
            print("Invalid response from GuessingAgent")
            return

        if confidence >= 9:
            print("High confidence guess, proceeding to make guess")
            yield self.create_text_response_event(json.dumps({
                "action": "make_guess",
                "guess": guess_output.get("guess"),
                "reasoning": guess_output.get("reasoning")
            }), invocation_id=invocation_id)
            return

        print("Low confidence guess, asking a question instead")

        async for event in self.asking_agent.run_async(ctx):
            yield event

        question_output = ctx.session.state.get("question_output", None)

        if question_output is None:
            print("Invalid response from AskingAgent")
            return

        yield self.create_text_response_event(json.dumps({
            "action": "ask_question",
            "question": question_output.get("question"),
            "reasoning": question_output.get("reasoning")
        }), invocation_id=invocation_id)
        return

root_agent = RootAgent(
    name="root_agent",
    guessing_agent=guessing_agent,
    asking_agent=asking_agent
)

Next, we will create the root session and the runner in a similar way as we did for the simple agent.

In [ ]:
# Function to create the root session

session_service = InMemorySessionService()
app_name = "RootAgent"

async def create_root_session():
  root_session = await session_service.create_session(
      app_name=app_name,
      user_id=user_id
  )
  return root_session


In [ ]:
# Create the agent runner

root_agent_runner = Runner(
    agent=root_agent,
    session_service=session_service,
    app_name=app_name
)

Then we will create the necessary UI widgets and a callback function.

In [ ]:
# Function to create the UI widgets

def create_question_ui():
  # Text display of the question
  question_label = widgets.Label("Welcome to the 20 Questions Game")

  # Yes button
  yes_button = widgets.Button(
      description='Yes',
      disabled=True,
      button_style='success', # Green for Yes
      tooltip='Click for Yes',
      icon='check'
  )

  # No button
  no_button = widgets.Button(
      description='No',
      disabled=True,
      button_style='danger', # Red for No
      tooltip='Click for No',
      icon='times'
  )

  # Group Yes and No buttons in an HBox to place them side-by-side
  buttons_box = widgets.HBox([yes_button, no_button])

  # Text display of the explanation
  explanation_label = widgets.Label("Thinking...")

  return question_label, buttons_box, explanation_label


In [ ]:
# Function to call the root agent using the runner

async def guess_or_ask(game_context: str = "Starting a new game of 20 questions"):
  try:
      async for event in root_agent_runner.run_async(
          user_id=user_id,
          session_id=root_session.id,
          new_message=types.Content(role='user', parts=[types.Part(text=game_context)])
      ):
          try:
              response = json.loads(event.content.parts[0].text)
              if "action" in response:
                  print(f"Response: {response}")
                  return response
          except Exception as e:
              print(f"Error parsing JSON response: {e}")
              continue
  except Exception as e:
      print(f"Error during guess_or_ask: {e}")
      return None

Let's put everything together!

In [ ]:
from IPython.lib.display import IFrame
root_session = asyncio.run(create_root_session())

question_label, buttons_box, explanation_label=create_question_ui()

last_question = None
question_counter = 0
max_questions = 20

# Display the UI for the first time
question_ui = widgets.VBox([question_label, buttons_box, explanation_label])

display(question_ui)

async def game_step(user_answer: str = None):

  global last_question,question_counter,max_questions

  if question_counter >= max_questions:
    question_label.value = "Game Over: AI loses"
    explanation_label.value = ""
    buttons_box.children[0].disabled = True # Disable Yes button
    buttons_box.children[1].disabled = True # Disable No button
    return

  if user_answer=='yes' and last_question is None:
    question_label.value = "Game Over: AI wins"
    explanation_label.value = ""
    return

  try:

    if last_question is not None and user_answer is not None:
      game_context = f"The answer to the last question, '{last_question}', was {user_answer}."
    elif user_answer is not None:
      game_context = "Let's continue and try again"
    else:
      game_context = "Starting a new game of 20 questions"

    print(game_context)
    question_counter += 1
    print(f"Question {question_counter}/{max_questions}")

    response = await guess_or_ask(game_context)

    if response and response.get("action") == "ask_question":
      question_label.value = f'Question {question_counter}/{max_questions}: {response.get("question")}'
      explanation_label.value = f'Reasoning: {response.get("reasoning")}'
      last_question = response.get("question")

    elif response and response.get("action") == "make_guess":
      question_label.value = f'My guess is: {response.get("guess")}'
      explanation_label.value = f'Reasoning: {response.get("reasoning")}'
      last_question=None

    buttons_box.children[0].disabled = False # Enable Yes button
    buttons_box.children[1].disabled = False # Enable No button

  except Exception as e:
    display(f"Error during game step: {e}")
    buttons_box.children[0].disabled = True # Disable Yes button
    buttons_box.children[1].disabled = True # Disable No button

def on_yes_button_clicked(b):
    explanation_label.value="Thinking..."
    buttons_box.children[0].disabled = True # Disable Yes button
    buttons_box.children[1].disabled = True # Disable No button
    asyncio.run(game_step("yes"))

def on_no_button_clicked(b):
    explanation_label.value="Thinking..."
    buttons_box.children[0].disabled = True # Disable Yes button
    buttons_box.children[1].disabled = True # Disable No button
    asyncio.run(game_step("no"))

buttons_box.children[0].on_click(on_yes_button_clicked)
buttons_box.children[1].on_click(on_no_button_clicked)

# Start the first question
asyncio.run(game_step())
